In [ ]:

#Feature Engineering FHVVH
#Logestic Regaression to predict whether or not a tip >10% of Base Fare will be given
#Model Evaluation

# Number of executors
number_of_executors = sc._jsc.sc().getExecutorMemoryStatus().keySet().size()
print(f"number_of_executors {number_of_executors}")

s = sc._jsc.sc().getExecutorMemoryStatus().keys()
l = str(s).replace("Set(","").replace(")","").split(", ")

print(f"{len(l)} Nodes: {l}")
d = set()
for i in l:
    d.add(i.split(":")[0])
len(d)

# Set global constants for Google Cloud Storage
BUCKET_NAME = 'my-bigdata-project-kc'
FIGURE_FOLDER = 'figures'

#Import Python modules needed for visualization
import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage
from pyspark.ml.stat import Correlation

# Import PySpark functions and libraries
from pyspark.sql.functions import col, isnan, when, count, udf, year, month, date_format, round
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Bucketizer

from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import numpy as np
from pyspark.mllib.evaluation import MulticlassMetrics
import io
from google.cloud import storage

# Initialize Spark session
spark = SparkSession.builder.appName("ReadAllMonthsParquet").getOrCreate()

def read_all_parquet_files():
    bucket_name = 'my-bigdata-project-kc'
    storage_client = storage.Client()

    # List all blobs in the cleaned folder
    blobs = storage_client.list_blobs(bucket_name, prefix="cleaned/")

    # Get parquet files
    parquet_blobs = [blob for blob in blobs if blob.name.endswith('.parquet')]

    df_list = []

    for blob in parquet_blobs:
        cleaned_filename = f"gs://{bucket_name}/{blob.name}"

        # Read the Parquet file into a temporary Spark DataFrame
        sdf_temp = spark.read.parquet(cleaned_filename)

        # Add to the list
        df_list.append(sdf_temp)

        # Print success message
        print(f"Successfully added: {blob.name}")

    # Combine DataFrames
    if df_list:
        sdf = df_list[0]
        for df in df_list[1:]:
            sdf = sdf.union(df)
    else:
        sdf = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema=None)
        print("No valid Parquet files found in the folder.")

    return sdf

# Run the function
sdf = read_all_parquet_files()

# Feature Engineering 1: Drop Unnecessary Columns
columns_to_drop = [
    "bcf", "sales_tax", "congestion_surcharge", "airport_fee", "driver_pay",
    "access_a_ride_flag", "wav_request_flag", "wav_match_flag",
    "request_datetime_year", "request_datetime_month", "request_datetime_day_of_month",
    "request_datetime_week_day", "request_datetime_hour", "request_datetime_minute",
    "request_datetime_time_of_day", "pickup_datetime_year", "pickup_datetime_minute",
    "pickup_datetime_hour","shared_match_flag", "tolls","pickup_datetime_month",
    "pickup_datetime_day_of_month", "tolls"
]
sdf = sdf.drop(*columns_to_drop)

# Feature Engineering 2: Create Interactive Feature "Tip Percentage"
sdf = sdf.withColumn("tip_percentage", (col("tips") / col("base_passenger_fare")) * 100)

# Create Tip Buckets
splits = [0.0, 10.0, 20.0, float("inf")]
tip_bucketizer = Bucketizer(splits=splits, inputCol="tip_percentage", outputCol="tip_bucket")
sdf = tip_bucketizer.transform(sdf)

# Create Binary label: 1 if good/great tip (>=10%), else 0
sdf = sdf.withColumn("label", when(col("tip_bucket").isin(1.0, 2.0), 1.0).otherwise(0.0))

#Feature Engineering 3: Create Interactive Features
# Trip_Weekend 1 if PickupDay is Saturday/Sunday, 0 otherwise
sdf = sdf.withColumn("trip_weekend", when((col("pickup_datetime_week_day") == '6') | (col("pickup_datetime_week_day") == '7'), 1.0).otherwise(0.0))

# Create Binary Feature PM=1, AM=0
sdf = sdf.withColumn("pickup_time_binary", when(col("pickup_datetime_time_of_day") == "pm", 1.0).otherwise(0.0))

# Convert Trip Duration to minutes
sdf = sdf.withColumn("trip_duration", col("trip_time") / 60)

# Feature Engineering 4: Convert Categorical Variables to Integers
sdf = sdf.withColumn("PULocationID", col("PULocationID").cast(IntegerType())) \
         .withColumn("DOLocationID", col("DOLocationID").cast(IntegerType())) \

         # Feature Engineering 5: Replace NAs with O
sdf_filled = sdf.fillna({
    "tip_percentage": 0,
    "trip_duration": 0,
    "trip_weekend": 0,
    "pickup_time_binary": 0,
    "trip_miles": 0
})

# Save Transformed df as a Parquet file in GCS Trusted folder

# Save Transformed df as a Parquet file in GCS Trusted folder
gcs_path = f"gs://{BUCKET_NAME}/trusted/"
sdf_filled.write.mode("overwrite").parquet(gcs_path)

# Print success message
print("Transformed data successfully saved to GCS as a Parquet file.")

# Split the data into 70% training and 30% test sets
train_data, test_data = sdf_filled.randomSplit([0.7, 0.3], seed=42)

# Pipeline Stages
pu_indexer = StringIndexer(inputCol="PULocationID", outputCol="PULocationID_index", handleInvalid="skip")
do_indexer = StringIndexer(inputCol="DOLocationID", outputCol="DOLocationID_index", handleInvalid="skip")

encoder = OneHotEncoder(inputCols=["PULocationID_index", "DOLocationID_index"],
                        outputCols=["PULocationID_vec", "DOLocationID_vec"], handleInvalid="keep")

assembler = VectorAssembler(inputCols=[ "trip_miles","trip_duration", "trip_weekend", "pickup_time_binary",
                                       "PULocationID_vec", "DOLocationID_vec"], outputCol="features", handleInvalid="skip")

#Create the Pipeline
pipeline = Pipeline(stages=[pu_indexer, do_indexer, encoder, assembler])

#Fit and Transform Training Datasets
pipeline_model = pipeline.fit(train_data)
train_transformed = pipeline_model.transform(train_data)
test_transformed = pipeline_model.transform(test_data)

# Train and Fit Logistic Regression
log_reg = LogisticRegression(featuresCol="features", labelCol="label",
                             predictionCol="custom_prediction", rawPredictionCol="custom_rawPrediction")

log_model = log_reg.fit(train_transformed)

from pyspark.ml.feature import OneHotEncoderModel

# Extract OneHotEncoderModel from the pipeline
encoder_model = [stage for stage in pipeline_model.stages if isinstance(stage, OneHotEncoderModel)][0]

# Get number of encoded features
pu_size = encoder_model.categorySizes[0]
do_size = encoder_model.categorySizes[1]

# Feature names used in the assembler
#base_features = ["trip_duration", "trip_weekend", "pickup_time_binary"]
base_features=["trip_miles","trip_duration", "trip_weekend", "pickup_time_binary",
                                       "PULocationID_vec", "DOLocationID_vec"]

# Coefficients from model
coefficients = log_model.coefficients

# Print coefficients only
print("Base Feature Coefficients:")
for name, coef in zip(base_features, coefficients[:len(base_features)]):
    print(f"  {name}: {coef:.4f}")

# Make Predictions
test_results = log_model.transform(test_transformed)

# Show the predicted tip
test_results.select('PULocationID', 'trip_duration', 'base_passenger_fare', 'tips', 'custom_prediction').show(truncate=False)




# Evaluation of AUC
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="custom_prediction", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(test_results)
print(f"Linear Model Area Under ROC: {auc:.4f}")

# Additional Metrics for binary classification
metrics = {
    "Accuracy": "accuracy",
    "F1 Score": "f1",
    "Precision": "weightedPrecision",
    "Recall": "weightedRecall"
}

# Using MulticlassClassificationEvaluator for metrics evaluation
for name, metric in metrics.items():
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="custom_prediction", metricName=metric)
    score = evaluator.evaluate(test_results)
    print(f"{name}: {score:.4f}")


#Save Pipeline and Logistic Models to the GCS Bucket

# Define GCS paths for saving the models
logistic_regression_model_path = "gs://my-bigdata-project-kc/models/FHVHV_logistic_regression_model_20250422"
pipeline_model_path = "gs://my-bigdata-project-kc/models/FHVHV_pipeline_model_20250422"

# Save the trained Logistic Regression model (use log_model instead of lr_model)
log_model.write().overwrite().save(logistic_regression_model_path)

# Save the trained Pipeline model (includes preprocessing steps and logistic regression model)
pipeline_model.write().overwrite().save(pipeline_model_path)

# Print confirmation messages
print(f"Logistic Regression model saved to: {logistic_regression_model_path}")
print(f"Pipeline model saved to: {pipeline_model_path}")




 # Frequency Distribution Table

# Count the frequency of each tip bucket
bucket_counts = sdf.groupBy("tip_bucket").count().orderBy("tip_bucket")

# Define bucket labels for clarity
bucket_labels = {
    0.0: "Low Tip (<10%)",
    1.0: "Good Tip (10%-20%)",
    2.0: "Great Tip (>20%)"
}

# Map the bucket values to the labels and filter out unknowns
bucket_counts_with_labels = (
    bucket_counts.rdd
    .filter(lambda row: row["tip_bucket"] in bucket_labels)
    .map(lambda row: (
        bucket_labels[row["tip_bucket"]],
        row["count"]
    ))
    .toDF(["Tip Bucket", "Count"])
)

# Show the frequency distribution
bucket_counts_with_labels.show(truncate=False)

# Convert to pandas for plotting
bucket_counts_pandas = bucket_counts_with_labels.toPandas()

# Plot the frequency distribution
plt.figure(figsize=(8, 6))
plt.bar(bucket_counts_pandas['Tip Bucket'], bucket_counts_pandas['Count'], color='skyblue')
plt.title('Frequency Distribution of Tip Buckets')
plt.xlabel('Tip Bucket')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()

import io
import matplotlib.pyplot as plt
from google.cloud import storage

# --- Save Plot to Google Cloud Storage (GCS) ---

# Create an in-memory binary stream to hold the plot
img_data = io.BytesIO()
plt.savefig(img_data, format='png', bbox_inches='tight')  # Save the current plot to the buffer
img_data.seek(0)  # Important: Reset the stream's position to the start

# Connect to GCS
storage_client = storage.Client()
bucket_name = 'my-bigdata-project-kc'
bucket = storage_client.get_bucket(bucket_name)

# Define full path where the image will be stored
FIGURE_FOLDER = "plots"  # example folder
filename = "frequency_dist_tip_buckets.png"
blob_path = f"{FIGURE_FOLDER}/{filename}"
blob = bucket.blob(blob_path)

# Upload the image buffer content to GCS
blob.upload_from_file(img_data, content_type='image/png')

print(f"Plot successfully uploaded to: gs://{bucket_name}/{blob_path}")










# Create the ML Pipeline
# Fit Feature Engineered SDF
# Transform SDF then Save to GCS


#Import Python modules needed for visualization
import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage
from pyspark.ml.stat import Correlation
# Import PySpark functions and libraries
from pyspark.sql.functions import col, isnan, when, count, udf, year, month, date_format, round,rand
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Bucketizer
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import numpy as np
from pyspark.mllib.evaluation import MulticlassMetrics
import io
from google.cloud import storage
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder
import itertools

# Initialize Spark session
spark = SparkSession.builder.appName("ReadAllMonthsParquet").getOrCreate()


# Set global constants for Google Cloud Storage
BUCKET_NAME = 'my-bigdata-project-kc'


#Get Feature Engineered SDF
def read_all_parquet_files():
    bucket_name = 'my-bigdata-project-kc'
    storage_client = storage.Client()

    # List all blobs in the cleaned folder
    blobs = storage_client.list_blobs(bucket_name, prefix="trusted/")

    # Get parquet files
    parquet_blobs = [blob for blob in blobs if blob.name.endswith('.parquet')]

    df_list = []

    for blob in parquet_blobs:
        cleaned_filename = f"gs://{bucket_name}/{blob.name}"

        # Read the Parquet file into a temporary Spark DataFrame
        sdf_temp = spark.read.parquet(cleaned_filename)

        # Add to the list
        df_list.append(sdf_temp)

        # Print success message
        print(f"Successfully added: {blob.name}")

    # Combine DataFrames
    if df_list:
        sdf = df_list[0]
        for df in df_list[1:]:
            sdf = sdf.union(df)
    else:
        sdf = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema=None)
        print("No valid Parquet files found in the folder.")

    return sdf

# Run the function
sdf_filled = read_all_parquet_files()

#ASSEMBLE AND SAVE PIPELINE FEATURES TO GCS

# Define feature transformation stages
pu_indexer = StringIndexer(inputCol="PULocationID", outputCol="PULocationID_index", handleInvalid="skip")
do_indexer = StringIndexer(inputCol="DOLocationID", outputCol="DOLocationID_index", handleInvalid="skip")

encoder = OneHotEncoder(inputCols=["PULocationID_index", "DOLocationID_index"],
                        outputCols=["PULocationID_vec", "DOLocationID_vec"], handleInvalid="keep")

assembler = VectorAssembler(inputCols=["trip_duration", "trip_weekend", "pickup_time_binary",
                                       "PULocationID_vec", "DOLocationID_vec"],
                            outputCol="features", handleInvalid="skip")

# Create pipeline without model
feature_pipeline = Pipeline(stages=[pu_indexer, do_indexer, encoder, assembler])

# Fit and transform
feature_model = feature_pipeline.fit(sdf_filled)
sdf_features = feature_model.transform(sdf_filled)

# Save the transformed DataFrame to GCS
sdf_features.write.mode("overwrite").parquet(f"gs://{BUCKET_NAME}/trusted/assembled_features")
print("Assembled features saved to GCS.")



















# Cross-Validation Setup and Evaluation
# Retrieve the assembled Spark DataFrame (features vector and label)
# Apply k-fold cross-validation for model tuning and evaluation
# Evaluate model performance using Area Under the ROC Curve (AUC)



# Import libraries
import io
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Google Cloud Storage
from google.cloud import storage

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, when, count, udf, year, month, date_format, round, rand
from pyspark.sql.types import IntegerType

#ML libraries
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Bucketizer
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.stat import Correlation
from pyspark.mllib.evaluation import MulticlassMetrics

import itertools

# Initialize Spark session
spark = SparkSession.builder.appName("ReadAllMonthsParquet").getOrCreate()

# Set global constants for Google Cloud Storage
BUCKET_NAME = 'my-bigdata-project-kc'

# Load the transformed dataset with assembled 'features' from GCS
sdf_loaded = spark.read.parquet(f"gs://{BUCKET_NAME}/trusted/assembled_features")

# Define your model
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
lr = LogisticRegression(featuresCol="features", labelCol="label", predictionCol="custom_prediction")

# Evaluator

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")


param_grid = ParamGridBuilder().build()

# CrossValidator
crossval = CrossValidator(
    estimator=lr,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5
)

# Fit model
cv_model = crossval.fit(sdf_loaded)
best_model = cv_model.bestModel

# Predict and evaluate
predictions = best_model.transform(sdf_loaded)
auc = evaluator.evaluate(predictions)
print(f"Cross-validated AUC: {auc:.4f}")

